In [1]:
library(Seurat)
library(scop)
library(Signac)
library(ggplot2)
set.seed(4180)
library(patchwork)
setwd("/")
#########color
cols <- c("#444576", "#4682B4",
 "#AEDEEE","#FFA500",
 "#FFD790","#C65762","#FBDFDE",
 "#F6EFCF","#BCB99F")
pal <- colorRampPalette(cols)

Loading required package: SeuratObject

Loading required package: sp




Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Registered S3 method overwritten by 'scop':
  method             from  
  FoldChange.default Seurat

          ⬢          .        ⬡             ⬢     .
                     _____ _________  ____
                    / ___// ___/ __ ./ __ .
                   (__  )/ /__/ /_/ / /_/ /
                  /____/ .___/.____/ .___/
                                  /_/
      ⬢               .      ⬡        .          ⬢

------------------------------------------------------------
Version: 0.8.7 (2026-03-31 update)
Website: https://mengxu98.github.io/scop/

Python environment initialization is disabled
To enable it, set: options(scop_env_init = TRUE)

The message can be suppressed by: 
  suppressPackageStartupMessages(library(scop))
  or options(log_message.verbose = FALSE)
------------------------------------------------------------



In [2]:
subsc.list <- readRDS("data/subsc.list.Rds")
names(subsc.list)

[1] "aCM"        "FB"         "EC"         "EndoCC"     "LEC"       
 [6] "SMC"        "Pericyte"   "Adipocyte"  "Neuronal"   "T"         
[11] "B"          "Macrophage" "DC"

In [3]:
ATAC <- readRDS("data/mHeart_ATAC.Rds")
activity <- readRDS("data/activity.Rds")

In [8]:
mHeart<- readRDS("data/mHeart.Rds")

In [10]:
#S2
cell.types <- c("vCM", "EC", "FB", "Macrophage", "aCM", "Adipocyte",
 "Pericyte", "Neuronal", "EndoCC", "SMC")
library(dplyr)
for(i in cell.types ) {
samplegroup <-c('NP', 'EP', 'MP', 'LP', 'PP')
DEGs_group <- data.frame()
obj <- subset(mHeart , celltype %in% i)
obj[["ATAC"]] <- ATAC[, colnames(obj)][["ATAC"]]
obj[["activity"]] <- activity[, colnames(obj)][["activity"]]
 for (j in 2:5){
 obj<- RunDEtest(srt = obj, group.by = "group",
 assay = 'ATAC',layer = 'counts',
 fc.threshold = 1.5, only.pos = FALSE, min.pct = 0.1,
 group1 = samplegroup[j],group2 = 'NP')
 DEGs_group <- rbind(DEGs_group,obj@tools$DEtest_custom$AllMarkers_wilcox)
 }
DEGs_group$col <- ifelse(DEGs_group$avg_log2FC>0,'up','down')
DEG_dt <- DEGs_group %>%
 group_by(group1, col) %>%
 summarise(count = n())
DEG_dt$count_adjusted <- ifelse(DEG_dt$col == "down", -DEG_dt$count, DEG_dt$count)
DEG_dt$col <- factor(DEG_dt$col, levels = c("up", "down"))
p<-ggplot(DEG_dt, aes(x = group1, y = count_adjusted, fill = col)) +
 geom_bar(stat = "identity", width = 0.8) +
 scale_fill_manual(values = rev(pal(4)[1:2])) + # colorsmust match up/down match
 geom_text(aes(label = count), # show raw value (non-negative)
 vjust = ifelse(DEG_dt$col == "up", -0.5, 1.2), # adjust text position
 color = "black", size = 3) +
 labs(
 title = "",
 x = "",
 y = "Number of differential peaks (.vs NP)",
 fill=''
 ) +
 theme_minimal() +
 theme(
 legend.position = 'right',
 panel.grid.major = element_blank(),
 panel.grid.minor = element_blank(),
 axis.line = element_line(color = "black"),
 axis.ticks = element_line(color = "black"),
 axis.text.x = element_text(angle = 0, hjust = 0.5)
 ) +
 scale_x_discrete(expand = c(0, 0)) +
 scale_y_continuous(
 expand = c(0, 0),
 limits = c(min(DEG_dt$count_adjusted ) - 2000, max(DEG_dt$count) + 2000), # symmetric range
 labels = abs # yaxis labels show absolute values
 )
ggsave(plot = p, filename = paste("S2_",i,"_left_diffPeaks.pdf",sep = ""),
 path = "plot/figureS2/", width = 3.5, height = 3,create.dir = T)
rna.dt <- AggregateExpression(obj, assays = 'RNA',group.by = 'group')[[1]]
act.dt <- AggregateExpression(obj, assays = 'activity',group.by = 'group')[[1]]
genes.dt <- intersect(row.names(rna.dt),row.names(act.dt))
p.dt <- data.frame(rna = rowSums(rna.dt[genes.dt,]),activities = rowSums(act.dt[genes.dt,]))
cor_value <- cor.test(log10(p.dt$rna + 1), log10(p.dt$activities + 1))
r <- round(cor_value$estimate, 3)
p_value <- scales::pvalue(cor_value$p.value)
p<-ggplot(data = p.dt, mapping = aes(x = log10(rna+1), y = log10(activities+1))) +
 geom_point(color='gray80', size=0.1) +
 geom_smooth(method = "lm", color = "#4682B4", formula = y ~ x) + # add linear regression line
 theme_light() +
 theme(
 panel.grid.major = element_blank(),
 panel.grid.minor = element_blank(),
 axis.line = element_line(color = "black"),
 legend.position = 'none'
 ) +
 xlab('Normalized RNA counts') + # fix axis label
 ylab('Normalized gene activities') +
 annotate("text", 
 x = Inf, y = 0, # place text in upper-right corner
 label = paste0("r = ", r, "\n", "p = ", p_value), 
 hjust = 1.1, vjust = 0.1, # fine-tune text position
 size = 4, color = "black")
ggsave(plot = p, filename = paste("S2_",i,"_right_cortest.pdf",sep = ""),path = "plot/figureS2/", width = 3, height = 3)
}

ℹ [2026-05-08 00:31:10] Data type is raw counts

ℹ [2026-05-08 00:31:10] Start differential expression test

ℹ [2026-05-08 00:31:10] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-08 00:31:16] Differential expression test completed

ℹ [2026-05-08 00:31:29] Data type is raw counts

ℹ [2026-05-08 00:31:29] Start differential expression test

ℹ [2026-05-08 00:31:29] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-08 00:31:37] Differential expression test completed

ℹ [2026-05-08 00:31:50] Data type is raw counts

ℹ [2026-05-08 00:31:50] Start differential expression test

ℹ [2026-05-08 00:31:50] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-08 00:31:56] Differential expression test completed

ℹ [2026-05-08 00:32:09] Data type is raw counts

ℹ [2026-05-08 00:32:09] Start differential expression test

ℹ [2026-05-08 00:32:09] Find all markers(wilcox) for custom cell groups...

✔ [2026-05-08 00:32:16] Differential expression test completed

`sum